# Chapter 1: Introduction to AI - Implementation

This notebook contains all Python implementations, visualizations, and hands-on examples from Chapter 1.

## What You'll Learn

- Build simple AI agents (reflex, model-based, goal-based)
- Implement basic learning algorithms (supervised, unsupervised, reinforcement)
- Work with genetic algorithms
- Visualize AI concepts

**Prerequisites**: Basic Python knowledge

**Reference**: See [ch01_introduction.ipynb](ch01_introduction.ipynb) for theory.

## Setup

Import required libraries for our implementations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

# For reproducibility
np.random.seed(42)
random.seed(42)

print("✓ Libraries imported successfully!")

## 1. Simple Reflex Agent

A **reflex agent** makes decisions based only on the current percept (observation).

### Example: Thermostat Agent

The agent reads room temperature and decides to turn heater on/off.

In [ ]:
class ThermostatAgent:
    """Simple reflex agent that controls room temperature."""
    
    def __init__(self, target_temp=22):
        self.target_temp = target_temp
    
    def perceive(self, room_temp):
        """Get current room temperature."""
        return room_temp
    
    def act(self, room_temp):
        """Decide action based on current temperature."""
        if room_temp < self.target_temp:
            return "HEAT_ON"
        else:
            return "HEAT_OFF"

# Test the agent
agent = ThermostatAgent(target_temp=22)

print("Testing Thermostat Agent:")
print(f"Room at 18°C → {agent.act(18)}")
print(f"Room at 25°C → {agent.act(25)}")
print(f"Room at 22°C → {agent.act(22)}")

### Visualization: Agent Behavior Over Time

In [ ]:
# Simulate room temperature over time
def simulate_thermostat(initial_temp, target_temp, steps=50):
    """Simulate thermostat agent controlling room temperature."""
    agent = ThermostatAgent(target_temp)
    
    temps = [initial_temp]
    actions = []
    
    current_temp = initial_temp
    
    for _ in range(steps):
        action = agent.act(current_temp)
        actions.append(1 if action == "HEAT_ON" else 0)
        
        # Simple temperature dynamics
        if action == "HEAT_ON":
            current_temp += 0.5  # Heating
        else:
            current_temp -= 0.3  # Cooling
        
        temps.append(current_temp)
    
    return temps, actions

# Run simulation
temps, actions = simulate_thermostat(initial_temp=18, target_temp=22)

# Plot results
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Temperature over time
ax1.plot(temps, 'b-', linewidth=2, label='Room Temperature')
ax1.axhline(y=22, color='r', linestyle='--', label='Target (22°C)')
ax1.set_ylabel('Temperature (°C)', fontsize=11)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title('Thermostat Agent Simulation', fontsize=13, fontweight='bold')

# Actions over time
ax2.fill_between(range(len(actions)), actions, alpha=0.5, color='orange')
ax2.set_xlabel('Time Steps', fontsize=11)
ax2.set_ylabel('Heater State', fontsize=11)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['OFF', 'ON'])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final temperature: {temps[-1]:.1f}°C")

## 2. Model-Based Agent

A **model-based agent** maintains an internal state to track the world.

### Example: Vacuum Cleaner Agent

The agent remembers which rooms it has cleaned.

In [ ]:
class VacuumAgent:
    """Model-based vacuum cleaner agent."""
    
    def __init__(self, rooms):
        self.rooms = rooms  # List of room names
        self.current_room = rooms[0]
        # Internal model: track cleaned status
        self.cleaned = {room: False for room in rooms}
    
    def perceive(self, room_dirty):
        """Check if current room is dirty."""
        return room_dirty
    
    def act(self, room_dirty):
        """Decide action based on percept and internal state."""
        if room_dirty and not self.cleaned[self.current_room]:
            self.cleaned[self.current_room] = True
            return "CLEAN"
        elif all(self.cleaned.values()):
            return "DONE"
        else:
            # Move to next uncleaned room
            for room in self.rooms:
                if not self.cleaned[room]:
                    self.current_room = room
                    return f"MOVE_TO_{room}"
        return "IDLE"

# Test the agent
rooms = ['A', 'B', 'C']
agent = VacuumAgent(rooms)

# Simulate cleaning
print("Vacuum Agent Cleaning Process:")
room_status = {'A': True, 'B': True, 'C': True}  # All dirty

step = 1
while True:
    action = agent.act(room_status[agent.current_room])
    print(f"Step {step}: Room {agent.current_room} → {action}")
    
    if action == "DONE":
        break
    if action == "CLEAN":
        room_status[agent.current_room] = False
    
    step += 1
    if step > 10:  # Safety limit
        break

print(f"\nFinal state: {agent.cleaned}")

## 3. Supervised Learning: Linear Regression

Learn a function from labeled examples.

### Example: House Price Prediction

Given house size, predict price.

In [ ]:
# Generate synthetic data
np.random.seed(42)
X = np.array([50, 60, 70, 80, 90, 100, 110, 120, 130, 140])  # Size (m²)
y = np.array([150, 180, 200, 230, 250, 280, 300, 330, 350, 380])  # Price (k$)
y = y + np.random.normal(0, 10, len(y))  # Add noise

# Simple linear regression: y = w*x + b
def fit_linear_regression(X, y):
    """Fit line using least squares."""
    n = len(X)
    # Calculate slope and intercept
    w = (n * np.sum(X * y) - np.sum(X) * np.sum(y)) / (n * np.sum(X**2) - np.sum(X)**2)
    b = (np.sum(y) - w * np.sum(X)) / n
    return w, b

w, b = fit_linear_regression(X, y)
print(f"Learned model: Price = {w:.2f} × Size + {b:.2f}")

# Predictions
y_pred = w * X + b

# Calculate error
mse = np.mean((y - y_pred)**2)
print(f"Mean Squared Error: {mse:.2f}")

### Visualization: Training Data and Fitted Line

In [ ]:
plt.figure(figsize=(10, 6))

# Scatter plot of data
plt.scatter(X, y, color='blue', s=100, alpha=0.6, label='Training Data', edgecolors='black')

# Fitted line
X_line = np.linspace(X.min(), X.max(), 100)
y_line = w * X_line + b
plt.plot(X_line, y_line, 'r-', linewidth=2, label=f'Fitted Line: y={w:.2f}x+{b:.2f}')

plt.xlabel('House Size (m²)', fontsize=12)
plt.ylabel('Price (k$)', fontsize=12)
plt.title('Linear Regression: House Price Prediction', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Make a prediction
new_size = 95
predicted_price = w * new_size + b
print(f"\nPrediction: A {new_size}m² house should cost ${predicted_price:.1f}k")

## 4. Unsupervised Learning: K-Means Clustering

Group data points without labels.

### Example: Customer Segmentation

Cluster customers by age and spending.

In [ ]:
# Generate synthetic customer data
np.random.seed(42)

# Three customer groups
group1 = np.random.randn(30, 2) * 5 + [25, 30]   # Young, low spending
group2 = np.random.randn(30, 2) * 5 + [45, 70]   # Middle-aged, high spending
group3 = np.random.randn(30, 2) * 5 + [60, 40]   # Senior, medium spending

X = np.vstack([group1, group2, group3])
print(f"Generated {len(X)} customer data points")

def kmeans(X, k, max_iters=10):
    """Simple K-means clustering."""
    # Random initialization
    centroids = X[np.random.choice(len(X), k, replace=False)]
    
    for iteration in range(max_iters):
        # Assign points to nearest centroid
        distances = np.sqrt(((X - centroids[:, np.newaxis])**2).sum(axis=2))
        labels = np.argmin(distances, axis=0)
        
        # Update centroids
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        
        # Check convergence
        if np.allclose(centroids, new_centroids):
            print(f"Converged in {iteration+1} iterations")
            break
        
        centroids = new_centroids
    
    return labels, centroids

# Run K-means
labels, centroids = kmeans(X, k=3)
print(f"Found {len(centroids)} clusters")

### Visualization: Customer Clusters

In [ ]:
plt.figure(figsize=(10, 6))

# Plot each cluster with different color
colors = ['red', 'blue', 'green']
for i in range(3):
    cluster_points = X[labels == i]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], 
                c=colors[i], label=f'Cluster {i+1}', alpha=0.6, s=50)

# Plot centroids
plt.scatter(centroids[:, 0], centroids[:, 1], 
            c='black', marker='X', s=300, label='Centroids', 
            edgecolors='yellow', linewidths=2)

plt.xlabel('Age', fontsize=12)
plt.ylabel('Spending Score', fontsize=12)
plt.title('K-Means Clustering: Customer Segmentation', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Analyze clusters
for i in range(3):
    cluster_points = X[labels == i]
    print(f"Cluster {i+1}: {len(cluster_points)} customers, "
          f"Avg Age={cluster_points[:, 0].mean():.1f}, "
          f"Avg Spending={cluster_points[:, 1].mean():.1f}")

## 5. Reinforcement Learning: Multi-Armed Bandit

Learn by trial and error with rewards.

### Example: Epsilon-Greedy Algorithm

Choose between multiple slot machines (arms) to maximize reward.

In [ ]:
class MultiArmedBandit:
    """Simple multi-armed bandit environment."""
    
    def __init__(self, true_rewards):
        self.true_rewards = true_rewards  # True mean reward for each arm
        self.n_arms = len(true_rewards)
    
    def pull(self, arm):
        """Pull an arm and get reward (with noise)."""
        return self.true_rewards[arm] + np.random.randn() * 0.1

class EpsilonGreedyAgent:
    """Agent using epsilon-greedy strategy."""
    
    def __init__(self, n_arms, epsilon=0.1):
        self.n_arms = n_arms
        self.epsilon = epsilon
        # Estimates of arm values
        self.Q = np.zeros(n_arms)
        # Number of times each arm was pulled
        self.N = np.zeros(n_arms)
    
    def select_arm(self):
        """Choose arm using epsilon-greedy."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_arms)  # Explore
        else:
            return np.argmax(self.Q)  # Exploit
    
    def update(self, arm, reward):
        """Update estimate after pulling arm."""
        self.N[arm] += 1
        # Incremental average
        self.Q[arm] += (reward - self.Q[arm]) / self.N[arm]

# Setup
true_rewards = [1.0, 2.0, 1.5, 3.0]  # Arm 3 is best
bandit = MultiArmedBandit(true_rewards)
agent = EpsilonGreedyAgent(n_arms=4, epsilon=0.1)

# Run experiment
n_steps = 1000
rewards_history = []
optimal_actions = []

for step in range(n_steps):
    arm = agent.select_arm()
    reward = bandit.pull(arm)
    agent.update(arm, reward)
    
    rewards_history.append(reward)
    optimal_actions.append(1 if arm == 3 else 0)  # Arm 3 is optimal

print(f"\nLearned arm values: {agent.Q}")
print(f"True arm values: {true_rewards}")
print(f"Best arm selected: Arm {np.argmax(agent.Q)}")
print(f"Optimal action rate: {np.mean(optimal_actions[-100:]):.1%}")

### Visualization: Learning Progress

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Average reward over time
window = 50
avg_rewards = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
ax1.plot(avg_rewards, linewidth=2)
ax1.axhline(y=max(true_rewards), color='r', linestyle='--', label='Optimal Reward')
ax1.set_xlabel('Steps', fontsize=11)
ax1.set_ylabel('Average Reward', fontsize=11)
ax1.set_title('Learning Progress', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Arm selection frequency
arm_counts = agent.N
ax2.bar(range(len(arm_counts)), arm_counts, color=['red', 'blue', 'green', 'orange'])
ax2.set_xlabel('Arm', fontsize=11)
ax2.set_ylabel('Times Selected', fontsize=11)
ax2.set_title('Arm Selection Frequency', fontsize=12, fontweight='bold')
ax2.set_xticks(range(4))
ax2.set_xticklabels([f'Arm {i}\n(r={true_rewards[i]:.1f})' for i in range(4)])
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Genetic Algorithm: Function Optimization

Evolve solutions using selection, crossover, and mutation.

### Example: Optimize f(x) = -(x-5)² + 10

Find x that maximizes the function.

In [ ]:
def fitness(x):
    """Fitness function to maximize."""
    return -(x - 5)**2 + 10

def create_population(size, bounds):
    """Create initial random population."""
    return np.random.uniform(bounds[0], bounds[1], size)

def select_parents(population, fitnesses, n_parents):
    """Select best individuals as parents."""
    indices = np.argsort(fitnesses)[-n_parents:]
    return population[indices]

def crossover(parent1, parent2):
    """Average crossover."""
    return (parent1 + parent2) / 2

def mutate(individual, mutation_rate=0.1, bounds=(0, 10)):
    """Add random noise with some probability."""
    if np.random.random() < mutation_rate:
        individual += np.random.randn() * 0.5
        # Keep within bounds
        individual = np.clip(individual, bounds[0], bounds[1])
    return individual

# Genetic Algorithm
pop_size = 20
n_generations = 30
bounds = (0, 10)

population = create_population(pop_size, bounds)
best_fitness_history = []

for gen in range(n_generations):
    # Evaluate fitness
    fitnesses = np.array([fitness(ind) for ind in population])
    best_fitness_history.append(fitnesses.max())
    
    # Selection
    parents = select_parents(population, fitnesses, n_parents=10)
    
    # Create next generation
    next_gen = list(parents)  # Keep best parents (elitism)
    
    while len(next_gen) < pop_size:
        # Crossover
        p1, p2 = np.random.choice(parents, 2, replace=False)
        child = crossover(p1, p2)
        # Mutation
        child = mutate(child)
        next_gen.append(child)
    
    population = np.array(next_gen)

# Final results
final_fitnesses = np.array([fitness(ind) for ind in population])
best_individual = population[np.argmax(final_fitnesses)]
best_fitness = final_fitnesses.max()

print(f"Best solution: x = {best_individual:.3f}")
print(f"Best fitness: {best_fitness:.3f}")
print(f"True optimum: x = 5.0, fitness = 10.0")

### Visualization: Evolution Progress

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Evolution of best fitness
ax1.plot(best_fitness_history, 'b-', linewidth=2, marker='o')
ax1.axhline(y=10, color='r', linestyle='--', label='True Optimum')
ax1.set_xlabel('Generation', fontsize=11)
ax1.set_ylabel('Best Fitness', fontsize=11)
ax1.set_title('Genetic Algorithm Progress', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Final population distribution
x_vals = np.linspace(0, 10, 100)
y_vals = [fitness(x) for x in x_vals]

ax2.plot(x_vals, y_vals, 'k-', linewidth=2, label='Fitness Function')
ax2.scatter(population, [fitness(x) for x in population], 
            c='red', s=100, alpha=0.6, label='Final Population', edgecolors='black')
ax2.scatter([5], [10], c='green', s=200, marker='*', 
            label='True Optimum', edgecolors='black', linewidths=2)
ax2.set_xlabel('x', fontsize=11)
ax2.set_ylabel('f(x)', fontsize=11)
ax2.set_title('Population Distribution', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Programming Tasks

### Task 1: Enhanced Reflex Agent (Easy)
Modify the thermostat agent to include a hysteresis band (don't switch heater on/off too frequently).

### Task 2: Goal-Based Agent (Medium)
Extend the vacuum agent to plan a path that minimizes total distance traveled.

### Task 3: Polynomial Regression (Medium)
Extend the linear regression to fit a polynomial curve (degree 2 or 3) to non-linear data.

### Task 4: Hierarchical Clustering (Medium)
Implement agglomerative hierarchical clustering and visualize the dendrogram.

### Task 5: Q-Learning (Hard)
Implement Q-learning for a simple grid world where the agent learns to navigate to a goal.

### Task 6: TSP with Genetic Algorithm (Hard)
Use a genetic algorithm to solve the Traveling Salesperson Problem with order crossover.

## Summary

In this notebook, you implemented:

1. **Reflex Agent**: Thermostat with simple condition-action rules
2. **Model-Based Agent**: Vacuum cleaner with internal state
3. **Supervised Learning**: Linear regression for predictions
4. **Unsupervised Learning**: K-means for clustering
5. **Reinforcement Learning**: Multi-armed bandit with epsilon-greedy
6. **Genetic Algorithm**: Function optimization with evolution

### Key Takeaways

- Agents perceive and act based on different architectures
- Learning algorithms find patterns in data or through experience
- Evolutionary approaches can solve optimization problems
- Visualization helps understand algorithm behavior

### Next Steps

- Chapter 2: Search algorithms (BFS, DFS, A*)
- Try the programming tasks above
- Experiment with different parameters and datasets